<a href="https://colab.research.google.com/github/brad1379/Movie-Recommendation-System/blob/master/Movie_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Popularity-Based Filtering

### Load the data

In [55]:
import pandas as pd

movies = pd.read_csv("movies.csv")
credits = pd.read_csv("credits.csv")
ratings = pd.read_csv("ratings.csv")

### Calculate a weighted rating

$$WR = \frac{v}{v + m}* R + \frac{m}{v + m}\ * C$$


v = number of votes for a movie \
m = minimum number of votes required \
R = average rating of the movie \
C = average rating across all movies


In [56]:
m = movies["vote_count"].quantile(0.9)
print(m)

1838.4000000000015


In [57]:
movies_filtered = movies.copy().loc[movies['vote_count'] >= m]

In [58]:
C = movies['vote_average'].mean()
print(C)

6.092171559442016


In [59]:
def weighted_rating(df, m=m, C=C):
  R = df['vote_average']
  v = df['vote_count']
  wr = (((v/(v+m)) * R) + ((m/(v+m)) * C))
  return wr

In [60]:
movies_filtered['weighted_rating'] = movies_filtered.apply(weighted_rating, axis=1)

In [61]:
movies_filtered.sort_values('weighted_rating', ascending=False)[["title", "weighted_rating"]].head(10)

,title,weighted_rating
1881,The Shawshank Redemption,8.059258
662,Fight Club,7.939256
65,The Dark Knight,7.920020
3232,Pulp Fiction,7.904645
96,Inception,7.863239
3337,The Godfather,7.851236
95,Interstellar,7.809479
809,Forrest Gump,7.803188
329,The Lord of the Rings: The Return of the King,7.727243
1990,The Empire Strikes Back,7.697884


# Content-Based Filtering

In [62]:
import pandas as pd
import csv

In [63]:
movies = pd.read_csv("movies_small.csv", delimiter=';', quotechar='"', quoting=csv.QUOTE_MINIMAL, engine='python')

In [64]:
movies

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,190000000,"[{""id"": 28, ""name"": ""Action""}]",http://www.furious7.com/,168259,"[{""id"": 830, ""name"": ""car race""}, {""id"": 3428,...",en,Furious 7,Deckard Shaw seeks revenge against Dominic Tor...,102.322217,"[{""name"": ""Universal Pictures"", ""id"": 33}, {""n...","[{""iso_3166_1"": ""JP"", ""name"": ""Japan""}, {""iso_...",2015-04-01,1506249360,137,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Vengeance Hits Home,Furious 7,7.3,4176
1,200000000,"[{""id"": 16, ""name"": ""Animation""}, {""id"": 10751...",http://www.disney.go.com/cars/,49013,"[{""id"": 830, ""name"": ""car race""}, {""id"": 9663,...",en,Cars 2,Star race car Lightning McQueen and his pal Ma...,49.986590,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2011-06-11,559852396,106,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Ka-ciao!,Cars 2,5.8,2033
2,170000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 878, ""na...",http://marvel.com/guardians,118340,"[{""id"": 8828, ""name"": ""marvel comic""}, {""id"": ...",en,Guardians of the Galaxy,"Light years from Earth, 26 years after being a...",481.098624,"[{""name"": ""Marvel Studios"", ""id"": 420}, {""name...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2014-07-30,773328629,121,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,All heroes start somewhere.,Guardians of the Galaxy,7.9,9742
3,145000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.kungfupanda.com/,140300,"[{""id"": 478, ""name"": ""china""}, {""id"": 779, ""na...",en,Kung Fu Panda 3,"Continuing his ""legendary adventures of awesom...",56.747978,"[{""name"": ""Twentieth Century Fox Film Corporat...","[{""iso_3166_1"": ""CN"", ""name"": ""China""}, {""iso_...",2016-01-23,521170825,95,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,Grab destiny by the rice dumplings.,Kung Fu Panda 3,6.7,1603
4,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
5,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [65]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [66]:
tfidf = TfidfVectorizer(stop_words='english')
movies['overview'] = movies['overview'].fillna("")

In [67]:
tfidf_matrix = tfidf.fit_transform(movies['overview'])
pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())

,26,abducted,accuser,adventure,adventures,assumes,attorney,awesomeness,bane,barsoom,...,terrorist,threats,toretto,transported,villainous,wanted,war,weary,world,years
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,...,0.00000,0.000000,0.333333,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.190097,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.155883,0.000000
2,0.237761,0.237761,0.237761,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.00000,0.237761,0.000000,0.000000,0.000000,0.389934
3,0.000000,0.000000,0.000000,0.000000,0.270444,0.00000,0.000000,0.270444,0.00000,0.000000,...,0.00000,0.270444,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.13565,0.271301,0.000000,0.13565,0.000000,...,0.13565,0.000000,0.000000,0.000000,0.13565,0.000000,0.000000,0.000000,0.000000,0.111235
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.353458,...,0.00000,0.000000,0.000000,0.176729,0.00000,0.000000,0.176729,0.176729,0.144920,0.000000


In [68]:
tfidf_matrix.shape

(6, 121)

### Similarity Matrix

In [69]:
from sklearn.metrics.pairwise import linear_kernel

similarity_matrix = linear_kernel(tfidf_matrix, tfidf_matrix)

similarity_matrix

array([[1.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [0.        , 1.        , 0.        , 0.        , 0.        ,
        0.02259057],
       [0.        , 0.        , 1.        , 0.        , 0.0433744 ,
        0.        ],
       [0.        , 0.        , 0.        , 1.        , 0.        ,
        0.03213867],
       [0.        , 0.        , 0.0433744 , 0.        , 1.        ,
        0.01612024],
       [0.        , 0.02259057, 0.        , 0.03213867, 0.01612024,
        1.        ]])

### Find most similar movies to a certain movie

In [70]:
movie_title = "John Carter"

In [71]:
idx = movies.loc[movies['title'] == movie_title].index[0]
idx

np.int64(5)

In [72]:
scores = list(enumerate(similarity_matrix[idx]))
scores = [(score[0], float(score[1])) for score in scores]
sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
sorted_scores

[(5, 1.0000000000000007),
 (3, 0.032138674066915646),
 (1, 0.02259056579532686),
 (4, 0.016120240648257757),
 (0, 0.0),
 (2, 0.0)]

In [73]:
movie_indices = [tpl[0] for tpl in sorted_scores[1:4]]
movie_indices

[3, 1, 4]

In [75]:
movies['title'].iloc[movie_indices]

,title
3,Kung Fu Panda 3
1,Cars 2
4,The Dark Knight Rises


In [76]:
def similar_movies(movie_title, nr_movies):
  idx = movies.loc[movies['title'] == movie_title].index[0]
  scores = list(enumerate(similarity_matrix[idx]))
  scores = [(score[0], float(score[1])) for score in scores]
  sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
  movie_indices = [tpl[0] for tpl in sorted_scores[1:nr_movies+1]]
  similar_titles = movies['title'].iloc[movie_indices]
  return similar_titles

In [77]:
similar_movies("Kung Fu Panda 3", 3)

,title
5,John Carter
0,Furious 7
1,Cars 2
